# IUROPA network consistency checks

This notebook checks whether the citation network is internally consistent with the decision/node universe.

It tests:

1. whether every `citing_decision_id` in `citations.csv` exists in `decisions_curated.csv`;
2. whether every `cited_decision_id` exists in `decisions_curated.csv`;
3. whether any citation is a self-citation;
4. whether cited decisions occur strictly before citing decisions;
5. how many citations are same-day or apparently future-dated;
6. what decision types / courts are involved in problematic observations.

No observations are dropped automatically. The notebook only diagnoses and exports problem tables for inspection.


In [1]:
# ============================================================
# 0. CONFIGURATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

CITATIONS_CSV = Path("citations.csv")
DECISIONS_CSV = Path("decisions_curated.csv")

for p in [CITATIONS_CSV, DECISIONS_CSV]:
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found. Put it in the same folder as this notebook."
        )

print("Inputs found:")
print(" ", CITATIONS_CSV.resolve())
print(" ", DECISIONS_CSV.resolve())


Inputs found:
  C:\Users\laze\Downloads\IUROPA\IUROPA\citations.csv
  C:\Users\laze\Downloads\IUROPA\IUROPA\decisions_curated.csv


In [2]:
# ============================================================
# 1. LOAD RAW TABLES
# ============================================================

citations = pd.read_csv(CITATIONS_CSV)
decisions = pd.read_csv(DECISIONS_CSV)

print("citations.csv shape:", citations.shape)
print("decisions_curated.csv shape:", decisions.shape)

print("\nCitation columns:")
print(citations.columns.tolist())

print("\nDecision columns:")
print(decisions.columns.tolist())


citations.csv shape: (283994, 2)
decisions_curated.csv shape: (56951, 12)

Citation columns:
['citing_decision_id', 'cited_decision_id']

Decision columns:
['decision_id', 'ecli', 'celex', 'decision_type', 'decision_date', 'court', 'case_id', 'case_number', 'proceeding_id', 'proceeding_number', 'proceeding_suffix', 'proceeding_name']


In [3]:
# ============================================================
# 2. IDENTIFY DECISION KEY / DATE / TYPE COLUMNS
# ============================================================

required_citation_cols = {"citing_decision_id", "cited_decision_id"}
missing = required_citation_cols - set(citations.columns)
if missing:
    raise KeyError(f"citations.csv missing required columns: {sorted(missing)}")

if "decision_id" not in decisions.columns:
    raise KeyError("decisions_curated.csv must contain decision_id.")

# Prefer these standard IUROPA columns where available.
date_col_candidates = [
    "decision_date",
    "date",
]

type_col_candidates = [
    "decision_type",
    "document_type",
]

court_col_candidates = [
    "court",
]

def first_existing(candidates, df):
    for c in candidates:
        if c in df.columns:
            return c
    return None

decision_date_col = first_existing(date_col_candidates, decisions)
decision_type_col = first_existing(type_col_candidates, decisions)
court_col = first_existing(court_col_candidates, decisions)

print("Detected decision date column:", decision_date_col)
print("Detected decision type column:", decision_type_col)
print("Detected court column:", court_col)

if decision_date_col is None:
    raise KeyError(
        "Could not identify a decision date column in decisions_curated.csv."
    )


Detected decision date column: decision_date
Detected decision type column: decision_type
Detected court column: court


In [4]:
# ============================================================
# 3. BASIC UNIVERSE CHECKS
# ============================================================

print("DECISION UNIVERSE")
print("-----------------")
print("Rows:", len(decisions))
print("Unique decision IDs:", decisions["decision_id"].nunique())
print("Duplicate decision IDs:", decisions["decision_id"].duplicated().sum())

print("\nCITATION UNIVERSE")
print("-----------------")
print("Rows:", len(citations))
print(
    "Unique citation pairs:",
    citations[["citing_decision_id", "cited_decision_id"]]
    .drop_duplicates()
    .shape[0]
)
print("Duplicate citation pairs:",
      citations.duplicated(["citing_decision_id", "cited_decision_id"]).sum())
print("Unique citing IDs:", citations["citing_decision_id"].nunique())
print("Unique cited IDs:", citations["cited_decision_id"].nunique())

if decision_type_col:
    print("\nDecision types:")
    display(
        decisions[decision_type_col]
        .value_counts(dropna=False)
        .rename_axis(decision_type_col)
        .reset_index(name="n")
    )

if court_col:
    print("\nCourts:")
    display(
        decisions[court_col]
        .value_counts(dropna=False)
        .rename_axis(court_col)
        .reset_index(name="n")
    )


DECISION UNIVERSE
-----------------
Rows: 56951
Unique decision IDs: 56951
Duplicate decision IDs: 0

CITATION UNIVERSE
-----------------
Rows: 283994
Unique citation pairs: 283994
Duplicate citation pairs: 0
Unique citing IDs: 23231
Unique cited IDs: 21747

Decision types:


,decision_type,n
0,judgment,25778
1,order,19835
2,AG opinion,11256
3,AG view,40
4,opinion,26
5,decision,15
6,ruling,1



Courts:


,court,n
0,Court of Justice,35436
1,General Court,19404
2,Civil Service Tribunal,2111


In [5]:
# ============================================================
# 4. DO ALL CITATION ENDPOINT IDs EXIST IN THE DECISION TABLE?
# ============================================================

decision_ids = set(decisions["decision_id"].dropna().astype(str))
citing_ids = set(citations["citing_decision_id"].dropna().astype(str))
cited_ids = set(citations["cited_decision_id"].dropna().astype(str))

missing_citing_ids = citing_ids - decision_ids
missing_cited_ids = cited_ids - decision_ids

print("Unique citing IDs missing from decisions_curated:", len(missing_citing_ids))
print("Unique cited IDs missing from decisions_curated:", len(missing_cited_ids))

citations["sender_in_decisions"] = citations["citing_decision_id"].astype(str).isin(decision_ids)
citations["receiver_in_decisions"] = citations["cited_decision_id"].astype(str).isin(decision_ids)
citations["both_endpoints_in_decisions"] = (
    citations["sender_in_decisions"] & citations["receiver_in_decisions"]
)

print("\nCitation rows:")
print("Sender found:", citations["sender_in_decisions"].sum(), "/", len(citations))
print("Receiver found:", citations["receiver_in_decisions"].sum(), "/", len(citations))
print("Both found:", citations["both_endpoints_in_decisions"].sum(), "/", len(citations))

missing_sender_rows = citations.loc[~citations["sender_in_decisions"]].copy()
missing_receiver_rows = citations.loc[~citations["receiver_in_decisions"]].copy()
missing_any_rows = citations.loc[~citations["both_endpoints_in_decisions"]].copy()

print("\nMissing sender row count:", len(missing_sender_rows))
print("Missing receiver row count:", len(missing_receiver_rows))
print("Missing either endpoint row count:", len(missing_any_rows))

if missing_citing_ids:
    print("\nExample missing citing IDs:")
    display(pd.DataFrame({"missing_citing_decision_id": sorted(missing_citing_ids)[:50]}))

if missing_cited_ids:
    print("\nExample missing cited IDs:")
    display(pd.DataFrame({"missing_cited_decision_id": sorted(missing_cited_ids)[:50]}))


Unique citing IDs missing from decisions_curated: 893
Unique cited IDs missing from decisions_curated: 35

Citation rows:
Sender found: 267095 / 283994
Receiver found: 283040 / 283994
Both found: 266217 / 283994

Missing sender row count: 16899
Missing receiver row count: 954
Missing either endpoint row count: 17777

Example missing citing IDs:


,missing_citing_decision_id
0,DOC_JUDG_CJ_1963_0070_19650407
1,DOC_JUDG_CJ_1963_0110_19660713
2,DOC_JUDG_CJ_1973_0041_19770309
3,DOC_JUDG_CJ_1988_0111_19900404
4,DOC_JUDG_CJ_1988_0143_19910221
5,DOC_JUDG_CJ_1988_0297_19901018
6,DOC_JUDG_CJ_1990_0037_20000127
7,DOC_JUDG_CJ_1991_0130_REV_19960116
8,DOC_JUDG_CJ_1995_0245_P_19990119
9,DOC_JUDG_CJ_1995_0288_P_20000406



Example missing cited IDs:


,missing_cited_decision_id
0,DOC_JUDG_CJ_2003_0486_20061214
1,DOC_JUDG_CJ_2004_0405_P_20070125
2,DOC_JUDG_CJ_2005_0080_P_20080911
3,DOC_JUDG_CJ_2006_0036_20080508
4,DOC_JUDG_CJ_2006_0121_P_20080909
5,DOC_JUDG_CJ_2006_0146_20070705
6,DOC_JUDG_CJ_2006_0148_20080515
7,DOC_JUDG_CJ_2006_0295_20080401
8,DOC_JUDG_CJ_2006_0513_P_20091006
9,DOC_JUDG_CJ_2007_0393_20090430


In [6]:
# ============================================================
# 5. MERGE BASIC DECISION METADATA ONTO SENDER + RECEIVER
# ============================================================

keep_cols = ["decision_id", decision_date_col]
if decision_type_col:
    keep_cols.append(decision_type_col)
if court_col:
    keep_cols.append(court_col)

# Keep a few additional useful identifiers if present.
for c in [
    "lead_proceeding_id",
    "proceeding_id",
    "case_id",
    "case_number",
    "proceeding_number",
    "proceeding_name",
]:
    if c in decisions.columns and c not in keep_cols:
        keep_cols.append(c)

lookup = decisions[keep_cols].drop_duplicates("decision_id").copy()

sender_lookup = lookup.rename(
    columns={
        c: f"citing_{c}"
        for c in lookup.columns
        if c != "decision_id"
    }
).rename(columns={"decision_id": "citing_decision_id"})

receiver_lookup = lookup.rename(
    columns={
        c: f"cited_{c}"
        for c in lookup.columns
        if c != "decision_id"
    }
).rename(columns={"decision_id": "cited_decision_id"})

citation_check = citations.merge(
    sender_lookup,
    on="citing_decision_id",
    how="left",
    validate="many_to_one",
)

citation_check = citation_check.merge(
    receiver_lookup,
    on="cited_decision_id",
    how="left",
    validate="many_to_one",
)

print("Merged citation-check table:", citation_check.shape)
display(citation_check.head())


Merged citation-check table: (283994, 21)


,citing_decision_id,cited_decision_id,sender_in_decisions,receiver_in_decisions,both_endpoints_in_decisions,citing_decision_date,citing_decision_type,citing_court,citing_proceeding_id,citing_case_id,...,citing_proceeding_number,citing_proceeding_name,cited_decision_date,cited_decision_type,cited_court,cited_proceeding_id,cited_case_id,cited_case_number,cited_proceeding_number,cited_proceeding_name
0,DOC_JUDG_CJ_1998_0082_P_20000525,DOC_JUDG_CJ_1995_0008_P_19980528,True,True,True,2000-05-25,judgment,Court of Justice,PROC_CJ_1998_0082_P,CASE_CJ_1998_0082,...,C-82/98 P,Kögler v Court of Justice,1998-05-28,judgment,Court of Justice,PROC_CJ_1995_0008_P,CASE_CJ_1995_0008,C-8/95,C-8/95 P,New Holland Ford Ltd v Commission of the Europ...
1,DOC_JUDG_CJ_1998_0082_P_20000525,DOC_JUDG_CJ_1983_0264_19850704,True,True,True,2000-05-25,judgment,Court of Justice,PROC_CJ_1998_0082_P,CASE_CJ_1998_0082,...,C-82/98 P,Kögler v Court of Justice,1985-07-04,judgment,Court of Justice,PROC_CJ_1983_0264,CASE_CJ_1983_0264,C-264/83,C-264/83,Delhez v Commission
2,DOC_JUDG_CJ_2018_0025_20190508,DOC_JUDG_CJ_2017_0308_20181115,True,True,True,2019-05-08,judgment,Court of Justice,PROC_CJ_2018_0025,CASE_CJ_2018_0025,...,C-25/18,Brian Andrew Kerr v Pavlo Postnov and Natalia ...,2018-11-15,judgment,Court of Justice,PROC_CJ_2017_0308,CASE_CJ_2017_0308,C-308/17,C-308/17,Hellenische Republik v Leo Kuhn
3,DOC_JUDG_CJ_2018_0025_20190508,DOC_JUDG_CJ_2016_0249_20170615,True,True,True,2019-05-08,judgment,Court of Justice,PROC_CJ_2018_0025,CASE_CJ_2018_0025,...,C-25/18,Brian Andrew Kerr v Pavlo Postnov and Natalia ...,2017-06-15,judgment,Court of Justice,PROC_CJ_2016_0249,CASE_CJ_2016_0249,C-249/16,C-249/16,Kareda
4,DOC_JUDG_CJ_2018_0025_20190508,DOC_JUDG_CJ_2016_0560_20180307,True,True,True,2019-05-08,judgment,Court of Justice,PROC_CJ_2018_0025,CASE_CJ_2018_0025,...,C-25/18,Brian Andrew Kerr v Pavlo Postnov and Natalia ...,2018-03-07,judgment,Court of Justice,PROC_CJ_2016_0560,CASE_CJ_2016_0560,C-560/16,C-560/16,E. ON Czech Holding AG v Michael Dĕdouch and O...


In [7]:
# ============================================================
# 6. TEMPORAL CONSISTENCY
# ============================================================

sender_date_col = f"citing_{decision_date_col}"
receiver_date_col = f"cited_{decision_date_col}"

citation_check[sender_date_col] = pd.to_datetime(
    citation_check[sender_date_col],
    errors="coerce"
)
citation_check[receiver_date_col] = pd.to_datetime(
    citation_check[receiver_date_col],
    errors="coerce"
)

citation_check["citation_lag_days"] = (
    citation_check[sender_date_col] - citation_check[receiver_date_col]
).dt.days

citation_check["is_self_citation"] = (
    citation_check["citing_decision_id"]
    == citation_check["cited_decision_id"]
)

citation_check["is_strict_past"] = (
    citation_check[receiver_date_col]
    < citation_check[sender_date_col]
)

citation_check["is_same_day"] = (
    citation_check[receiver_date_col]
    == citation_check[sender_date_col]
)

citation_check["is_future_citation"] = (
    citation_check[receiver_date_col]
    > citation_check[sender_date_col]
)

citation_check["missing_sender_date"] = citation_check[sender_date_col].isna()
citation_check["missing_receiver_date"] = citation_check[receiver_date_col].isna()

print("TEMPORAL CONSISTENCY")
print("--------------------")
print("Total citations:", len(citation_check))
print("Strictly past:", int(citation_check["is_strict_past"].sum()))
print("Same day:", int(citation_check["is_same_day"].sum()))
print("Future-dated:", int(citation_check["is_future_citation"].sum()))
print("Self-citations:", int(citation_check["is_self_citation"].sum()))
print("Missing sender date:", int(citation_check["missing_sender_date"].sum()))
print("Missing receiver date:", int(citation_check["missing_receiver_date"].sum()))

print("\nCitation lag summary (days):")
display(citation_check["citation_lag_days"].describe())


TEMPORAL CONSISTENCY
--------------------
Total citations: 283994
Strictly past: 260252
Same day: 5040
Future-dated: 925
Self-citations: 2805
Missing sender date: 16899
Missing receiver date: 954

Citation lag summary (days):


count    266217.000000
mean       2840.042725
std        2516.518374
min       -8364.000000
25%         960.000000
50%        2144.000000
75%        4026.000000
max       20944.000000
Name: citation_lag_days, dtype: float64

In [8]:
# ============================================================
# 7. PROBLEM TABLES
# ============================================================

same_day = citation_check.loc[citation_check["is_same_day"]].copy()
future = citation_check.loc[citation_check["is_future_citation"]].copy()
self_citations = citation_check.loc[citation_check["is_self_citation"]].copy()
non_strict_past = citation_check.loc[
    citation_check["citation_lag_days"].notna()
    & (citation_check["citation_lag_days"] <= 0)
].copy()

print("Same-day citations:", len(same_day))
print("Future-dated citations:", len(future))
print("Self-citations:", len(self_citations))
print("All lag <= 0:", len(non_strict_past))

cols_to_show = [
    c for c in [
        "citing_decision_id",
        "cited_decision_id",
        sender_date_col,
        receiver_date_col,
        "citation_lag_days",
        f"citing_{decision_type_col}" if decision_type_col else None,
        f"cited_{decision_type_col}" if decision_type_col else None,
        f"citing_{court_col}" if court_col else None,
        f"cited_{court_col}" if court_col else None,
    ]
    if c is not None and c in citation_check.columns
]

print("\nExamples: same-day")
display(same_day[cols_to_show].head(30))

print("\nExamples: future-dated")
display(future[cols_to_show].head(30))

print("\nExamples: self-citations")
display(self_citations[cols_to_show].head(30))


Same-day citations: 5040
Future-dated citations: 925
Self-citations: 2805
All lag <= 0: 5965

Examples: same-day


,citing_decision_id,cited_decision_id,citing_decision_date,cited_decision_date,citation_lag_days,citing_decision_type,cited_decision_type,citing_court,cited_court
49,DOC_JUDG_GC_2005_0456_20100428,DOC_JUDG_GC_2005_0456_20100428,2010-04-28,2010-04-28,0.0,judgment,judgment,General Court,General Court
124,DOC_JUDG_CJ_2002_0175_20050113,DOC_JUDG_CJ_2002_0174_20050113,2005-01-13,2005-01-13,0.0,judgment,judgment,Court of Justice,Court of Justice
240,DOC_JUDG_CJ_2022_0639_20240905,DOC_JUDG_CJ_2022_0639_20240905,2024-09-05,2024-09-05,0.0,judgment,judgment,Court of Justice,Court of Justice
287,DOC_JUDG_GC_1993_0432_19950307,DOC_JUDG_GC_1993_0432_19950307,1995-03-07,1995-03-07,0.0,judgment,judgment,General Court,General Court
365,DOC_JUDG_CJ_2018_0573_20191009,DOC_JUDG_CJ_2018_0573_20191009,2019-10-09,2019-10-09,0.0,judgment,judgment,Court of Justice,Court of Justice
384,DOC_JUDG_CJ_1982_0010_19830714,DOC_JUDG_CJ_1982_0010_19830714,1983-07-14,1983-07-14,0.0,judgment,judgment,Court of Justice,Court of Justice
399,DOC_JUDG_CJ_1987_0291_19881214,DOC_JUDG_CJ_1987_0291_19881214,1988-12-14,1988-12-14,0.0,judgment,judgment,Court of Justice,Court of Justice
400,DOC_JUDG_CJ_2024_0212_20250508,DOC_JUDG_CJ_2024_0212_20250508,2025-05-08,2025-05-08,0.0,judgment,judgment,Court of Justice,Court of Justice
429,DOC_JUDG_GC_2016_0836_20190516,DOC_JUDG_GC_2016_0836_20190516,2019-05-16,2019-05-16,0.0,judgment,judgment,General Court,General Court
459,DOC_JUDG_CST_2005_0022_20061213,DOC_JUDG_CST_2005_0022_20061213,2006-12-13,2006-12-13,0.0,judgment,judgment,Civil Service Tribunal,Civil Service Tribunal



Examples: future-dated


,citing_decision_id,cited_decision_id,citing_decision_date,cited_decision_date,citation_lag_days,citing_decision_type,cited_decision_type,citing_court,cited_court
1300,DOC_JUDG_CST_2007_0114_20090929,DOC_JUDG_GC_2009_0091_P_20100901,2009-09-29,2010-09-01,-337.0,judgment,judgment,Civil Service Tribunal,General Court
1376,DOC_JUDG_CST_2008_0052_20090604,DOC_JUDG_GC_2006_0404_P_20090908,2009-06-04,2009-09-08,-96.0,judgment,judgment,Civil Service Tribunal,General Court
1612,DOC_JUDG_CST_2014_0132_20151006,DOC_JUDG_GC_2015_0309_P_20161027,2015-10-06,2016-10-27,-387.0,judgment,judgment,Civil Service Tribunal,General Court
1884,DOC_JUDG_CST_2010_0062_20110915,DOC_JUDG_GC_2009_0325_P_20110921,2011-09-15,2011-09-21,-6.0,judgment,judgment,Civil Service Tribunal,General Court
1920,DOC_JUDG_GC_2016_0639_P_20180123,DOC_JUDG_GC_2016_0750_20181214,2018-01-23,2018-12-14,-325.0,judgment,judgment,General Court,General Court
2444,DOC_JUDG_GC_1993_0485_19960924,DOC_JUDG_GC_1993_0485_20001108,1996-09-24,2000-11-08,-1506.0,judgment,judgment,General Court,General Court
2447,DOC_JUDG_GC_1993_0485_19960924,DOC_JUDG_CJ_1996_0386_P_19980505,1996-09-24,1998-05-05,-588.0,judgment,judgment,General Court,Court of Justice
2448,DOC_JUDG_GC_1993_0485_19960924,DOC_JUDG_CJ_1996_0391_P_19980505,1996-09-24,1998-05-05,-588.0,judgment,judgment,General Court,Court of Justice
2449,DOC_JUDG_GC_1993_0485_19960924,DOC_JUDG_CJ_1996_0403_P_19980505,1996-09-24,1998-05-05,-588.0,judgment,judgment,General Court,Court of Justice
2454,DOC_JUDG_GC_1993_0485_19960924,DOC_JUDG_GC_1997_0266_19990708,1996-09-24,1999-07-08,-1017.0,judgment,judgment,General Court,General Court



Examples: self-citations


,citing_decision_id,cited_decision_id,citing_decision_date,cited_decision_date,citation_lag_days,citing_decision_type,cited_decision_type,citing_court,cited_court
49,DOC_JUDG_GC_2005_0456_20100428,DOC_JUDG_GC_2005_0456_20100428,2010-04-28,2010-04-28,0.0,judgment,judgment,General Court,General Court
240,DOC_JUDG_CJ_2022_0639_20240905,DOC_JUDG_CJ_2022_0639_20240905,2024-09-05,2024-09-05,0.0,judgment,judgment,Court of Justice,Court of Justice
287,DOC_JUDG_GC_1993_0432_19950307,DOC_JUDG_GC_1993_0432_19950307,1995-03-07,1995-03-07,0.0,judgment,judgment,General Court,General Court
365,DOC_JUDG_CJ_2018_0573_20191009,DOC_JUDG_CJ_2018_0573_20191009,2019-10-09,2019-10-09,0.0,judgment,judgment,Court of Justice,Court of Justice
384,DOC_JUDG_CJ_1982_0010_19830714,DOC_JUDG_CJ_1982_0010_19830714,1983-07-14,1983-07-14,0.0,judgment,judgment,Court of Justice,Court of Justice
399,DOC_JUDG_CJ_1987_0291_19881214,DOC_JUDG_CJ_1987_0291_19881214,1988-12-14,1988-12-14,0.0,judgment,judgment,Court of Justice,Court of Justice
400,DOC_JUDG_CJ_2024_0212_20250508,DOC_JUDG_CJ_2024_0212_20250508,2025-05-08,2025-05-08,0.0,judgment,judgment,Court of Justice,Court of Justice
429,DOC_JUDG_GC_2016_0836_20190516,DOC_JUDG_GC_2016_0836_20190516,2019-05-16,2019-05-16,0.0,judgment,judgment,General Court,General Court
459,DOC_JUDG_CST_2005_0022_20061213,DOC_JUDG_CST_2005_0022_20061213,2006-12-13,2006-12-13,0.0,judgment,judgment,Civil Service Tribunal,Civil Service Tribunal
523,DOC_JUDG_GC_2013_0429_20180517,DOC_JUDG_GC_2013_0429_20180517,2018-05-17,2018-05-17,0.0,judgment,judgment,General Court,General Court


In [9]:
# ============================================================
# 8. ARE PROBLEMS ASSOCIATED WITH PARTICULAR TYPES / COURTS?
# ============================================================

if decision_type_col:
    sender_type_col = f"citing_{decision_type_col}"
    receiver_type_col = f"cited_{decision_type_col}"

    print("ALL CITATIONS: sender x receiver decision type")
    display(
        pd.crosstab(
            citation_check[sender_type_col],
            citation_check[receiver_type_col],
            margins=True,
            dropna=False,
        )
    )

    if len(non_strict_past):
        print("\nNON-STRICT-PAST CITATIONS: sender x receiver decision type")
        display(
            pd.crosstab(
                non_strict_past[sender_type_col],
                non_strict_past[receiver_type_col],
                margins=True,
                dropna=False,
            )
        )

if court_col:
    sender_court_col = f"citing_{court_col}"
    receiver_court_col = f"cited_{court_col}"

    print("\nALL CITATIONS: sender x receiver court")
    display(
        pd.crosstab(
            citation_check[sender_court_col],
            citation_check[receiver_court_col],
            margins=True,
            dropna=False,
        )
    )

    if len(non_strict_past):
        print("\nNON-STRICT-PAST CITATIONS: sender x receiver court")
        display(
            pd.crosstab(
                non_strict_past[sender_court_col],
                non_strict_past[receiver_court_col],
                margins=True,
                dropna=False,
            )
        )


ALL CITATIONS: sender x receiver decision type


cited_decision_type,judgment,NaN,All
citing_decision_type,,,
judgment,266217,878,267095.0
NaN,16823,76,NaN
All,283040,0,283994.0



NON-STRICT-PAST CITATIONS: sender x receiver decision type


cited_decision_type,judgment,All
citing_decision_type,,
judgment,5965,5965
All,5965,5965



ALL CITATIONS: sender x receiver court


cited_court,Civil Service Tribunal,Court of Justice,General Court,NaN,All
citing_court,,,,,
Civil Service Tribunal,3188,1650,5524,17,10379.0
Court of Justice,31,88343,3303,360,92037.0
General Court,1628,63812,98738,501,164679.0
NaN,42,10806,5975,76,NaN
All,4889,164611,113540,0,283994.0



NON-STRICT-PAST CITATIONS: sender x receiver court


cited_court,Civil Service Tribunal,Court of Justice,General Court,All
citing_court,,,,
Civil Service Tribunal,281,14,192,487
Court of Justice,1,2315,97,2413
General Court,2,160,2903,3065
All,284,2489,3192,5965


In [10]:
# ============================================================
# 9. DEFINE A DIAGNOSTIC REM-VALID FLAG
# ============================================================
#
# This is ONLY a diagnostic flag for now.
# Nothing is dropped automatically.

citation_check["valid_rem_event_candidate"] = (
    citation_check["both_endpoints_in_decisions"]
    & ~citation_check["is_self_citation"]
    & citation_check["is_strict_past"]
)

summary = pd.DataFrame({
    "metric": [
        "total_citations",
        "both_endpoints_found",
        "self_citations",
        "same_day_citations",
        "future_dated_citations",
        "strict_past_citations",
        "valid_rem_event_candidate",
    ],
    "n": [
        len(citation_check),
        int(citation_check["both_endpoints_in_decisions"].sum()),
        int(citation_check["is_self_citation"].sum()),
        int(citation_check["is_same_day"].sum()),
        int(citation_check["is_future_citation"].sum()),
        int(citation_check["is_strict_past"].sum()),
        int(citation_check["valid_rem_event_candidate"].sum()),
    ],
})

summary["pct_of_all_citations"] = 100 * summary["n"] / len(citation_check)

display(summary)


,metric,n,pct_of_all_citations
0,total_citations,283994,100.000000
1,both_endpoints_found,266217,93.740361
2,self_citations,2805,0.987697
3,same_day_citations,5040,1.774685
4,future_dated_citations,925,0.325711
5,strict_past_citations,260252,91.639964
6,valid_rem_event_candidate,260252,91.639964


In [11]:
# ============================================================
# 10. OPTIONAL EXPORTS FOR INSPECTION
# ============================================================
#
# These are small CSV diagnostics, not cleaned production data.

summary.to_csv("network_consistency_summary.csv", index=False)

missing_any_rows.to_csv(
    "network_consistency_missing_endpoints.csv",
    index=False
)

non_strict_past.to_csv(
    "network_consistency_non_strict_past.csv",
    index=False
)

print("Saved:")
print(" network_consistency_summary.csv")
print(" network_consistency_missing_endpoints.csv")
print(" network_consistency_non_strict_past.csv")


Saved:
 network_consistency_summary.csv
 network_consistency_missing_endpoints.csv
 network_consistency_non_strict_past.csv
